In [0]:
## CRIAÇÃO DAS TABELAS DE BUSINESS E REVIEW COM APENAS REGISTROS DE RESTAURANTES

from pyspark.sql.functions import col

# Configuração
CATALOG = "workspace"
SCHEMA_BRONZE = "yelp_ing"
SCHEMA_SILVER = "yelp_ing"

# Carrega tabelas review e business já processadas
df_review = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_review")
df_business = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_business")
df_user = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_user")

print(f"Quantidade total de estabelecimentos: {df_business.count()}")

# Filtra food_category desejadas
food_categories = ["RESTAURANTE", "BAR E BEBIDA", "PADARIA", "CAFE"]
df_business_food = df_business.filter(col("food_category").isin(food_categories))
print(f"  Estabelecimentos nas categorias food: {df_business_food.count()}")

# ========== CRIA TABELA SILVER BUSINESS FILTERED ==========
print("\n--- Criando tabela silver_business_filtered ---")
table_silver_business_filtered = f"{CATALOG}.{SCHEMA_SILVER}.silver_business_filtered"
df_business_food.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_silver_business_filtered)
print(f"✓ Tabela silver_business_filtered criada: {table_silver_business_filtered}")
print(f"Total de registros silver_business_filtered: {df_business_food.count()}")

# Remove colunas desnecessárias da tabela silver_business_filtered
cols_to_drop = ["address", "attributes", "categories", "is_open", "latitude", "longitude", "postal_code"]
df_business_food_clean = df_business_food.drop(*cols_to_drop)

# Sobrescreve a tabela com as colunas removidas
df_business_food_clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_silver_business_filtered)
print(f"✓ Tabela silver_business_filtered atualizada sem colunas: {', '.join(cols_to_drop)}")
print(f"Total de registros silver_business_filtered: {df_business_food_clean.count()}")

print(f"Reviews totais de estabelecimentos: {df_review.count()}")

# Filtra reviews apenas de estabelecimentos de alimentação (food_category)
df_review_food = df_review.join(
    df_business_food.select("business_id", "food_category"),
    on="business_id",
    how="inner"
).filter(col("food_category").isin(food_categories))
print(f"  Reviews de estabelecimentos food: {df_review_food.count()}")

# ========== CRIA TABELA SILVER REVIEW FILTERED ==========
print("\n--- Criando tabela silver_review_filtered ---")
table_silver_review_filtered = f"{CATALOG}.{SCHEMA_SILVER}.silver_review_filtered"
df_review_food.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_silver_review_filtered)
print(f"✓ Tabela silver_review_filtered criada: {table_silver_review_filtered}")
print(f"Total de registros silver_review_filtered: {df_review_food.count()}")

# Remove coluna 'text' da tabela silver_review_filtered
df_review_food_clean = df_review_food.drop("text")
df_review_food_clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_silver_review_filtered)
print(f"✓ Tabela silver_review_filtered atualizada sem coluna: text")
print(f"Total de registros silver_review_filtered: {df_review_food_clean.count()}")



In [0]:
## CRIAÇÃO DA TABELA DE USERS COM APENAS REGISTROS DE RESTAURANTES

from pyspark.sql.functions import col

# Configuração
CATALOG = "workspace"
SCHEMA_BRONZE = "yelp_ing"
SCHEMA_SILVER = "yelp_ing"

# Carrega tabelas review e business já processadas
df_user = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_user")
table_silver_review_filtered = f"{CATALOG}.{SCHEMA_SILVER}.silver_review_filtered"

# Carrega tabelas silver
print(f"Total de usuários: {df_user.count()}")

# ========== CRIA TABELA SILVER USERS FILTERED ==========
print("\n--- Criando tabela silver_users_filtered ---")
table_silver_users_filtered = f"{CATALOG}.{SCHEMA_SILVER}.silver_users_filtered"

# Carrega user_ids presentes em silver_review_filtered
user_ids_food = spark.table(table_silver_review_filtered).select("user_id").distinct()

# Filtra df_user_silver apenas com user_ids presentes em silver_review_filtered
df_users_filtered = df_user.join(user_ids_food, on="user_id", how="inner")

df_users_filtered.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_silver_users_filtered)
print(f"✓ Tabela silver_users_filtered criada: {table_silver_users_filtered}")
print(f"Total de registros silver_users_filtered: {df_users_filtered.count()}")

# Remove colunas desnecessárias da tabela silver_users_filtered
cols_to_drop = [
    "compliment_cool", "compliment_cute", "compliment_funny", "compliment_hot",
    "compliment_list", "compliment_more", "compliment_note", "compliment_photos",
    "compliment_plain", "compliment_profile", "compliment_writer", "cool",
    "friends", "funny","useful"
]
df_users_filtered_clean = df_users_filtered.drop(*cols_to_drop)

# Sobrescreve a tabela com as colunas removidas
df_users_filtered_clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_silver_users_filtered)
print(f"✓ Tabela silver_users_filtered atualizada sem colunas: {', '.join(cols_to_drop)}")
print(f"Total de registros silver_users_filtered: {df_users_filtered_clean.count()}")

display(spark.table(table_silver_users_filtered).limit(5))